# 항공사 승객 만족도 예측 파이프라인 (Predicting Airline Satisfaction)

- **대회 (Competition)**: [Kaggle Playground Series S6E10 — Predicting Airline Satisfaction](https://www.kaggle.com/competitions/playground-series-s6e10)
- **작업 유형 (Task Type)**: 정형 데이터 이진 분류 (Binary Tabular Classification)
- **주요 지표 (Primary Metric)**: ROC-AUC (Area Under the Receiver Operating Characteristic Curve)

---

## 문제 정의 및 목표 (Problem Formulation)
- **목적 (Objective)**: 비행 운항 정보, 승객 인적사항, 기내 서비스 품질 평가 지표를 종합하여 승객의 최종 서비스 만족 여부(`satisfaction`)를 예측.
- **타깃 변수 (Target Variable)**: `satisfaction` (Binary: `satisfied` = 1, `neutral or dissatisfied` = 0)
- **입력 피처 (Input Features)**: 총 22개 피처 (인구통계학적 특성, 비행 거리/지연 시간, 14개 기내 서비스 세부 만족도 평가 점수 등)
- **예측 출력 (Model Output)**: 각 승객 ID별 만족 확률값 ($P(\text{satisfaction}=1) \in [0, 1]$)

## 평가 지표 및 산출 규격 (Evaluation & Submission Spec)
- **평가 지표 (Evaluation Metric)**: **ROC-AUC**
  - 분류 결정 임계값(Threshold)에 독립적인 랭킹 기반 판별력 측정 ($0.5$: 무작위 추정, $1.0$: 완벽한 분별)
  - 예측 확률값의 절대적 캘리브레이션(Calibration)보다 양성/음성 클래스 간 상대적 순위(Ranking) 보존이 핵심
- **제출 규격 (Submission Specification)**:
  - 스키마: `id`(승객 고유 식별자), `satisfaction`(만족 예측 확률, 부동소수점 $0.0 \sim 1.0$)
  - 결측 없이 전체 Test 샘플에 대한 확률값 산출 필수

## 파이프라인 아키텍처 (Pipeline Architecture)
1. **Data Ingestion & Integrity Check**: 런타임 환경(Windows, macOS, Linux, Kaggle) 자동 감지 및 입력 데이터셋 무결성 검증
2. **Exploratory Data Analysis (EDA)**: 타깃 클래스 균형, 결측치 분포, 범주형/수치형 특성 분포 및 타깃과의 상관관계 분석
3. **Benchmark Modeling**: 전처리 파이프라인 + 로지스틱 회귀(Logistic Regression) 기반 성능 기준선 수립
4. **Main Modeling & Validation**: Stratified 5-Fold Cross Validation 기반 LightGBM 학습, OOF(Out-of-Fold) 검증 (목표 AUC $\approx 0.958$) 및 Feature Importance 분석
5. **Inference & Artifact Generation**: Test 데이터셋 앙상블 추론 및 최종 `submission.csv` 산출

> **Kaggle 환경**
> 1. 대회 페이지 → *Code* → *New Notebook*
> 2. *File → Import Notebook* → `baseline.ipynb` 업로드
> 3. 우측 *Input* 패널에 `playground-series-s6e10` 데이터셋 마운트 확인
> 4. *Run All* 실행 후 *Output* 탭의 `submission.csv` 제출

---

## 1. 환경 설정 및 데이터 경로 탐색 (Environment & Path Setup)

- **재현성 보장 (Reproducibility)**: 전역 난수 시드(`SEED = 42`)를 설정하여 샘플링 및 모델 학습 결과 재현.
- **크로스 플랫폼 경로 자동 해결 (`find_data_dir`)**:
  - Windows, macOS, Linux 및 Kaggle Cloud 런타임 환경 자동 감지.
  - OS별 경로 구분자(`\` 및 `/`)와 무관하게 `pathlib.Path`를 통해 필수 데이터 파일(`train.csv`, `test.csv`, `sample_submission.csv`) 3종의 유효 디렉터리를 자동 탐색 및 무결성 검증.

In [ ]:
import time
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from matplotlib.ticker import PercentFormatter
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
START_TIME = time.time()

# Chart style: thin marks, light grid, no top/right frame.
COLOR_MAIN = "#2a78d6"   # blue   - single-series charts, "satisfied"
COLOR_ALT = "#eb6834"    # orange - "not satisfied"
plt.rcParams.update({
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3df",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "axes.titleweight": "bold",
})

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scikit-learn", sklearn.__version__)
print("lightgbm    ", lgb.__version__)

In [ ]:
REQUIRED_FILES = ["train.csv", "test.csv", "sample_submission.csv"]


def has_all_files(folder: Path) -> bool:
    return all((folder / name).is_file() for name in REQUIRED_FILES)


def find_data_dir() -> Path:
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates = sorted({p.parent for p in kaggle_root.glob("**/train.csv") if has_all_files(p.parent)})
    else:
        local = [p.resolve() for p in (Path("data"), Path("../data")) if has_all_files(p)]
        candidates = list(dict.fromkeys(local))  # drop duplicates, keep order

    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            f"데이터 폴더를 찾지 못했습니다. {REQUIRED_FILES} 세 파일이 모두 있는 폴더가 필요합니다.\n"
            "- Kaggle: 오른쪽 Input 패널에서 대회 데이터를 추가하세요.\n"
            "- 로컬: 노트북과 같은 위치의 data/ 폴더에 파일을 넣으세요."
        )
    raise RuntimeError(
        "데이터 폴더 후보가 여러 개입니다. 사용할 폴더를 DATA_DIR에 직접 지정하세요:\n"
        + "\n".join(f"  - {c}" for c in candidates)
    )


DATA_DIR = find_data_dir()
print("DATA_DIR =", DATA_DIR)

## 2. 데이터 수집 및 기초 스키마 검증 (Data Ingestion & Schema Inspection)

- `train.csv`: 타깃 레이블(`satisfaction`)이 포함된 지도학습 데이터셋
- `test.csv`: 모델 추론 대상 평가 데이터셋 (타깃 레이블 미포함)
- `sample_submission.csv`: 리더보드 제출 산출물 스키마 템플릿 (`id`, `satisfaction`)

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train :", train.shape)
print("test  :", test.shape)
print("sample:", sample.shape)
train.head()

In [ ]:
train.info()

### 📌 피처 스키마 분석 (Feature Schema Analysis)
- **식별자 (`id`)**: 행 고유 식별자로 모델 학습 피처에서 제외.
- **타깃 (`satisfaction`)**: 이진 분류 종속 변수 (Binary Label).
- **피처 군집**:
  - **기내 서비스 평가 지표 (13개)**: `Inflight wifi service`, `Seat comfort` 등 ($0 \sim 5$점 척도, 0은 Not Applicable/미응답).
  - **수치형 피처 (4개)**: `Age`, `Flight Distance`, `Departure Delay in Minutes`, `Arrival Delay in Minutes`.
  - **범주형 피처 (4개)**: `Gender`, `Customer Type`, `Type of Travel`, `Class`.

In [ ]:
TARGET = "satisfaction"
CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]
NUM_COLS = [c for c in train.columns if c not in CAT_COLS + ["id", TARGET]]
RATING_COLS = [c for c in NUM_COLS if train[c].max() <= 5]

print(f"범주형 {len(CAT_COLS)}개 :", CAT_COLS)
print(f"숫자형 {len(NUM_COLS)}개 (그중 평점 {len(RATING_COLS)}개)")

## 3. 탐색적 데이터 분석 (Exploratory Data Analysis)

피처별 분포 특성, 결측치 패턴, 타깃과의 상관관계를 분석하여 모델링 전략을 수립합니다.

### 3-1. 타깃 클래스 분포 (Target Class Distribution)

In [ ]:
target_rate = train[TARGET].value_counts(normalize=True).sort_index()
print(target_rate.round(4))

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Not satisfied", "Satisfied"], target_rate.values, color=[COLOR_ALT, COLOR_MAIN], width=0.5)
for i, v in enumerate(target_rate.values):
    ax.text(i, v + 0.01, f"{v:.1%}", ha="center")
ax.set_ylim(0, 0.7)
ax.set_ylabel("Share of passengers")
ax.set_title("Target balance")
plt.show()

- **클래스 비율**: Satisfied 약 44%, Neutral/Dissatisfied 약 56%.
- **데이터 처리 방향**: 심각한 클래스 불균형이 없으므로 추가적인 리샘플링 없이 계층화 분할(Stratified Split) 적용.

### 3-2. 결측치 분석 (Missing Value Analysis)

In [ ]:
missing = pd.DataFrame({"train": train.isna().sum(), "test": test.isna().sum()})
missing[(missing["train"] > 0) | (missing["test"] > 0)]

- **결측 현황**: `Arrival Delay in Minutes` 단일 피처에서만 결측치 발생 (전체의 0.05% 미만).
- **결측치 처리 전략**:
  - Tree 모델(LightGBM): 결측 분기 자동 학습 활용 (NaN 유지).
  - 선형 모델(Logistic Regression): 중앙값 대체(Median Imputation) 적용.

### 3-3. 범주형 피처와 타깃 간 분포 분석 (Categorical Feature vs Target)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), sharey=True)
for ax, col in zip(axes, CAT_COLS):
    rate = train.groupby(col)[TARGET].mean().sort_values()
    ax.bar(rate.index.astype(str), rate.values, color=COLOR_MAIN, width=0.5)
    for i, v in enumerate(rate.values):
        ax.text(i, v + 0.01, f"{v:.0%}", ha="center", fontsize=9)
    ax.set_title(col)
    ax.tick_params(axis="x", labelsize=9)
axes[0].set_ylabel("Satisfaction rate")
axes[0].set_ylim(0, 1)
plt.tight_layout()
plt.show()

- **범주형 피처 영향도**:
  - `Type of Travel` (출장 vs 개인 여행) 및 `Class` (Business vs Eco): 만족 비율 편차가 매우 커 핵심 판별 인자로 작용.
  - `Customer Type` (충성 고객 vs 비충성 고객): 유의미한 만족도 차이 확인.
  - `Gender`: 성별 간 만족 비율 편차가 미미하여 예측 기여도가 낮을 것으로 판단.

### 3-4. 기내 서비스 평가 점수와 만족도 간 상관관계 (Service Ratings vs Target)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, col in zip(axes, ["Online boarding", "Inflight wifi service"]):
    rate = train.groupby(col)[TARGET].mean()
    ax.bar(rate.index.astype(str), rate.values, color=COLOR_MAIN, width=0.6)
    ax.set_title(col)
    ax.set_xlabel("Rating (0-5)")
axes[0].set_ylabel("Satisfaction rate")
axes[0].set_ylim(0, 1)
plt.tight_layout()
plt.show()

- **비선형성 및 0점 특이 구간 분석**:
  - $1 \sim 5$점 구간: 평점 증가에 따라 만족 비율이 지수적으로 상승.
  - $0$점 특이 구간: `Online boarding` (약 62%), `Inflight wifi` (약 88%) 등에서 0점의 만족 비율이 역전 관측 (해당 서비스 미이용/해당 없음 도메인 특성 추정).
- **모델링 시사점**: U-shape 형태의 비선형 관계가 뚜렷하므로, 단순 선형 모델 대비 트리 기반 모델(GBDT)이 높은 적합도를 보일 것으로 예상.

In [ ]:
rating_corr = train[RATING_COLS].corrwith(train[TARGET].astype(int)).sort_values(ascending=False)
rating_corr.round(3).to_frame("corr_with_target")

### 3-5. 연속형 피처 분포 분석: 연령 및 비행 거리 (Age & Flight Distance)

In [ ]:
# Age is an integer, so use whole-number bin edges to avoid a saw-tooth pattern.
BINS = {"Age": np.arange(5, 91, 5), "Flight Distance": 50}

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for ax, col in zip(axes, ["Age", "Flight Distance"]):
    for value, label, color in [(False, "Not satisfied", COLOR_ALT), (True, "Satisfied", COLOR_MAIN)]:
        ax.hist(train.loc[train[TARGET] == value, col], bins=BINS[col], density=True,
                histtype="step", linewidth=2, color=color, label=label)
    ax.set_title(col)
    ax.set_ylabel("Density")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()

- **연령 (`Age`)**: $40 \sim 60$세 구간에서 만족 클래스 밀도가 높고, 40세 미만 및 60세 이상에서 불만족 밀도 우세.
- **비행 거리 (`Flight Distance`)**: 단거리($\le 1,000$) 구간은 불만족 비중이 높고, 장거리($\ge 2,000$) 구간은 비즈니스석 탑승 비중 증가와 연동되어 만족 비중 우세.

### 3-6. 운항 지연 시간 분포 (Delay Time Distribution)

In [ ]:
delay = train["Departure Delay in Minutes"]
print(f"출발 지연이 0분인 승객 비율: {(delay == 0).mean():.1%}")

delay_bucket = pd.cut(delay, bins=[-1, 0, 15, 60, np.inf], labels=["0", "1-15", "16-60", "60+"])
train.groupby(delay_bucket, observed=True)[TARGET].agg(passengers="size", satisfaction_rate="mean").round(3)

- **지연 특성**: 전체 표본의 90% 이상이 지연 0분(정시 운항)인 Zero-inflated 분포를 형성. 구간화(Binning) 분석 시 지연 시간 증가에 따라 만족 비율이 점진적 감소 경향.

### 3-7. 학습/평가 데이터셋 분포 정합성 검증 (Data Drift Check)

In [ ]:
rows = []
for col in CAT_COLS:
    tr_share = train[col].value_counts(normalize=True)
    te_share = test[col].value_counts(normalize=True)
    for level in tr_share.index:
        rows.append({"feature": col, "level": level,
                     "train_share": tr_share[level], "test_share": te_share.get(level, 0.0)})
share_table = pd.DataFrame(rows)
share_table["diff"] = share_table["test_share"] - share_table["train_share"]
share_table.round(4)

- **Covariate Shift 검증**: Train과 Test 데이터셋 간 모든 범주형 변수의 비율 오차가 0.2%p 이내로 분포 일관성 확보.

## 4. 데이터 전처리 파이프라인 (Data Preprocessing Pipeline)

모델 학습을 위한 피처 엔지니어링 및 타입 캐스팅 정의:
1. **타깃 인코딩 (Target Binarization)**: Boolean(`True/False`) $\to$ Integer(`1/0`)
2. **범주형 피처 처리 (Categorical Dtype Casting)**: LightGBM native 지원을 위해 pandas `category` 타입 변환 (Train/Test 일관된 범주셋 공유)
3. **피처 필터링 (Feature Filtering)**: 예측과 무관한 식별자(`id`) 제거
4. **결측치 처리 (Missing Values)**: LightGBM 분기 탐색 최적화를 위해 NaN 유지

In [ ]:
y = train[TARGET].astype(int)
X = train.drop(columns=["id", TARGET]).copy()
X_test = test.drop(columns=["id"]).copy()

for col in CAT_COLS:
    categories = sorted(train[col].unique())
    X[col] = pd.Categorical(X[col], categories=categories)
    X_test[col] = pd.Categorical(X_test[col], categories=categories)

assert list(X.columns) == list(X_test.columns)
print("X      :", X.shape)
print("X_test :", X_test.shape)
print("y mean :", round(y.mean(), 4))

## 5. 벤치마크 모델: 로지스틱 회귀 (Benchmark Baseline: Logistic Regression)

최소 성능 기준선(Performance Lower Bound)을 정의하기 위한 선형 분류 파이프라인 구성.

- **데이터 분할**: Train 80% / Validation 20% 계층화 분할 (`stratify=y`)
- **전처리 파이프라인 (`ColumnTransformer` & `Pipeline`)**:
  - 수치형 피처: Median Imputation $\to$ Standard Scaling
  - 범주형 피처: One-Hot Encoding (`handle_unknown='ignore'`)
- **데이터 누수 방지 (Data Leakage Prevention)**: 스케일러 및 통계량 추정은 학습용 세트(`X_tr`)에서만 Fit 수행.

In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
])
logreg = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])
logreg.fit(X_tr, y_tr)

va_pred = logreg.predict_proba(X_va)[:, 1]
logreg_auc = roc_auc_score(y_va, va_pred)
print(f"Logistic regression  AUC = {logreg_auc:.5f}   accuracy = {accuracy_score(y_va, va_pred > 0.5):.5f}")

- **벤치마크 결과**: 로지스틱 회귀 검증 ROC-AUC $\approx 0.920$ 달성. 후속 트리 모델 성능 검증을 위한 기준선으로 활용.

## 6. 메인 모델: LightGBM & Stratified 5-Fold 교차검증

LightGBM(Gradient Boosted Decision Trees)을 적용한 주 모델 학습 및 교차 검증 파이프라인.

### 교차 검증 및 앙상블 전략 (Cross-Validation Strategy)
- **Stratified 5-Fold Cross Validation**: 타깃 비율을 보존하며 5개 Fold로 분할.
- **Out-of-Fold (OOF) Prediction**: 전체 학습 세트에 대한 검증 예측치를 수집하여 과적합 없는 일반화 성능 추정.
- **Test Inference Ensemble**: 5개 Fold 모델의 소프트 보팅(Probability Average) 앙상블.
- **조기 종료 (Early Stopping)**: 100 iteration 동안 검증 AUC 개선 부재 시 조기 종료하여 과적합 방지.

In [ ]:
LGB_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "seed": SEED,
    "verbose": -1,
}
N_FOLDS = 5

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_pred = np.full(len(X), np.nan)
oof_count = np.zeros(len(X), dtype=int)
test_pred = np.zeros(len(X_test))
importance = pd.Series(0.0, index=X.columns)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    dtrain = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
    dvalid = lgb.Dataset(X.iloc[va_idx], label=y.iloc[va_idx], reference=dtrain)

    model = lgb.train(
        LGB_PARAMS,
        dtrain,
        num_boost_round=5000,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )

    oof_pred[va_idx] = model.predict(X.iloc[va_idx], num_iteration=model.best_iteration)
    oof_count[va_idx] += 1
    test_pred += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS
    importance += model.feature_importance(importance_type="gain") / N_FOLDS

    fold_auc = roc_auc_score(y.iloc[va_idx], oof_pred[va_idx])
    print(f"Fold {fold}: AUC = {fold_auc:.5f}   trees = {model.best_iteration}")

# Every training row must have been predicted exactly once.
assert (oof_count == 1).all()
assert np.isfinite(oof_pred).all()

lgb_auc = roc_auc_score(y, oof_pred)
print(f"\nLightGBM OOF AUC = {lgb_auc:.5f}   accuracy = {accuracy_score(y, oof_pred > 0.5):.5f}")

In [ ]:
pd.DataFrame({
    "model": ["Logistic regression (holdout 20%)", "LightGBM (5-fold OOF)"],
    "AUC": [logreg_auc, lgb_auc],
}).round(5)

- **성능 검증 결과**: OOF ROC-AUC $\approx 0.958$ (로지스틱 회귀 대비 약 $+0.038$p 개선).
- **개선 요인**: 서비스 평점 및 범주형 변수 간 고차 비선형 상호작용(High-order Interaction) 효과 학습. Fold 간 스코어 편차가 균일하여 모델 안정성 확인.

## 7. 피처 중요도 분석 (Feature Importance Analysis)

5개 Fold 모델의 Split Gain(손실 감소 기여도) 평균값을 기준으로 핵심 피처의 기여도를 정량화합니다.

In [ ]:
imp_share = (importance / importance.sum()).sort_values()

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(imp_share.index, imp_share.values, color=COLOR_MAIN, height=0.6)
for i, v in enumerate(imp_share.values):
    if v >= 0.02:
        ax.text(v + 0.005, i, f"{v:.0%}", va="center", fontsize=9)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Share of total gain")
ax.set_title("LightGBM feature importance (gain, mean of 5 folds)")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

- **주요 기여 피처 (Top Predictors)**: `Online boarding`, `Inflight wifi service`, `Type of Travel` 순으로 높은 Gain 점수 기록.
- **선형 상관계수와의 차이점**: `Inflight wifi service`는 단순 상관계수 순위(7위) 대비 트리 중요도(2위)로 급상승하여 비선형 패턴(0점 특이값 등)이 유효하게 반영되었음을 입증.
- **비기여 피처**: 타깃 편차가 작았던 `Gender` 등은 피처 기여도가 거의 0에 수렴.

## 8. 제출 파일 생성 및 정합성 검증 (Artifact Generation & Submission Validation)

- Test 데이터셋의 고유 ID와 5-Fold 앙상블 예측 확률값을 결합.
- `sample_submission.csv` 기준 행 수, ID 정렬, 결측치 유무, 확률 범위($[0, 1]$) 유효성을 사전 Assert 검증 후 `submission.csv` 저장.

In [ ]:
pred_df = pd.DataFrame({"id": test["id"].values, TARGET: test_pred})

assert pred_df["id"].is_unique
assert set(pred_df["id"]) == set(sample["id"])

submission = sample[["id"]].merge(pred_df, on="id", how="left", validate="one_to_one")

assert len(submission) == len(sample)
assert (submission["id"].values == sample["id"].values).all()
assert submission[TARGET].notna().all()
assert submission[TARGET].between(0, 1).all()

submission.to_csv("submission.csv", index=False)
print("saved:", Path("submission.csv").resolve())
print(submission.shape)
submission.head()

In [ ]:
print(f"전체 실행 시간: {time.time() - START_TIME:.0f}초")

## 9. 향후 개선 과제 (Engineering Roadmap & Iteration)

| 개선 영역 | 제안 전략 및 방법론 |
|---|---|
| **하이퍼파라미터 최적화** | Optuna 기반 Bayesian Optimization (`learning_rate`, `num_leaves`, `min_child_samples`, `colsample_bytree` 등) |
| **모델 아키텍처 다변화** | CatBoost, XGBoost 모델 학습 및 트리 알고리즘 간 앙상블 구성 |
| **앙상블 및 블렌딩** | OOF 예측값 기반 Ridge/LGBM 메타 모델(Stacking) 및 가중 평균(Weighted Blending) |
| **피처 엔지니어링** | 0점(Not Applicable) 여부 바이너리 플래그, 서비스 평점 통계치(Mean, Std, Sum), 순 지연시간(`Arrival - Departure Delay`) 피처 추가 |

<br>

> **실험 원칙 (Validation Protocol)**
> - 단일 변경 원칙(Single Variable Experimentation): 각 실험마다 단일 변경점만 적용하여 영향도 격리 분석.
> - OOF 점수 기준 평가: 리더보드 Public Test 스코어에 대한 과적합을 방지하고 로컬 OOF ROC-AUC를 모델 채택 기준으로 활용.